In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

DATA_PATH = "../Data Pack/"
OUTPUT_PATH = "../outputs/data_audit/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

competitions = pd.read_csv(DATA_PATH + "competitions.csv")
teams = pd.read_csv(DATA_PATH + "teams.csv")
matches = pd.read_csv(DATA_PATH + "matches.csv")
ball_by_ball = pd.read_csv(DATA_PATH + "ball_by_ball.csv")

player_mapping = pd.read_csv(
    DATA_PATH + "competition_team_player_mapping.csv"
)

playing_eleven = pd.read_csv(
    DATA_PATH + "playing_eleven.csv"
)

print("ALL SIX DATASETS LOADED")

datasets = {
    "competitions": competitions,
    "teams": teams,
    "matches": matches,
    "ball_by_ball": ball_by_ball,
    "player_mapping": player_mapping,
    "playing_eleven": playing_eleven
}

for name, df in datasets.items():
    print(f"{name:20} | Shape: {df.shape}")

ALL SIX DATASETS LOADED
competitions         | Shape: (141, 4)
teams                | Shape: (353, 3)
matches              | Shape: (883, 22)
ball_by_ball         | Shape: (200999, 33)
player_mapping       | Shape: (6413, 6)
playing_eleven       | Shape: (28381, 9)


In [2]:
print("BOWLING SPEED ANALYSIS")

print(ball_by_ball["bowl_speed"].describe())

print("\nSpeed values below zero:")

display(
    ball_by_ball[
        ball_by_ball["bowl_speed"] < 0
    ][["match_id", "bowler_name", "bowl_speed"]].head(20)
)

print("\nSpeed values above 160:")

display(
    ball_by_ball[
        ball_by_ball["bowl_speed"] > 160
    ][["match_id", "bowler_name", "bowl_speed"]]
    .sort_values("bowl_speed", ascending=False)
    .head(20)
)

print("\nZero speed records:")

print((ball_by_ball["bowl_speed"] == 0).sum())

BOWLING SPEED ANALYSIS
count    200999.000000
mean         71.413743
std          60.012263
min          -1.000000
25%           0.000000
50%          91.700000
75%         130.400000
max        1456.100000
Name: bowl_speed, dtype: float64

Speed values below zero:


,match_id,bowler_name,bowl_speed
6756,14108,Axar Patel,-1.0



Speed values above 160:


,match_id,bowler_name,bowl_speed
15236,14144,Khaleel Ahmed,1456.1
44885,19960,Sisanda Magala,1410.0
26821,16259,Jacob Duffy,1351.0
8782,14116,Kagiso Rabada,1339.6
10252,14122,Dwayne Bravo,1305.0
64721,21178,Benjamin Lister,1257.0
41809,17764,Mitchell Starc,1246.0
64443,21176,Dasun Shanaka,1229.7
1761,13524,Romario Shepherd,1135.6
64444,21176,Dasun Shanaka,1129.0



Zero speed records:
77861


In [3]:
print("EXTRAS DISTRIBUTION")

display(
    ball_by_ball["extras_type"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

print("\nEXTRAS RUN DISTRIBUTION")

display(
    ball_by_ball.groupby(
        "extras_type",
        dropna=False
    ).agg(
        deliveries=("match_id", "size"),
        extras1_total=("extras1_runs", "sum"),
        extras2_total=("extras_2_runs", "sum"),
        total_batter_runs=("runs", "sum")
    ).sort_values("deliveries", ascending=False)
)

EXTRAS DISTRIBUTION


,Count
extras_type,
NaN,189167
wide,7714
legbyes,2748
Overstep,576
byes,529
other,264
Penalty,1



EXTRAS RUN DISTRIBUTION


,deliveries,extras1_total,extras2_total,total_batter_runs
extras_type,,,,
NaN,189167,1,1,268285
wide,7714,9174,0,0
legbyes,2748,3603,0,1
Overstep,576,576,48,852
byes,529,1000,0,0
other,264,264,35,412
Penalty,1,5,0,0


In [4]:
print("DISMISSAL DISTRIBUTION")

display(
    ball_by_ball[
        ball_by_ball["is_wicket"] == True
    ]["dismissal_type"]
    .value_counts(dropna=False)
    .to_frame("Dismissal Count")
)

print("\nWICKET FLAG VALIDATION")

print("Total wickets:", ball_by_ball["is_wicket"].sum())

print(
    "Wickets with missing dismissal type:",
    ball_by_ball[
        (ball_by_ball["is_wicket"] == True)
        & ball_by_ball["dismissal_type"].isna()
    ].shape[0]
)

DISMISSAL DISTRIBUTION


,Dismissal Count
dismissal_type,
Caught,7557
Bowled,1775
LBW,709
RunOut,600
Stump,214
Hit Wicket,17
Retire Out,6
Obstracting the field,2
Obstructing the field,1



WICKET FLAG VALIDATION
Total wickets: 10881
Wickets with missing dismissal type: 0


In [5]:
print("PLAYING ELEVEN ANALYSIS")

display(
    pd.crosstab(
        playing_eleven["is_played"],
        playing_eleven["is_impact"],
        margins=True
    )
)

print("\nMISSING VALUES BY PLAYED STATUS")

display(
    playing_eleven.groupby("is_played").agg(
        total_players=("player_id", "size"),
        missing_batting_order=("batting_order",
                               lambda x: x.isna().sum()),
        missing_captain=("is_captain",
                         lambda x: x.isna().sum()),
        missing_wicketkeeper=("is_wk",
                              lambda x: x.isna().sum())
    )
)

PLAYING ELEVEN ANALYSIS


is_impact,0,1,All
is_played,,,
False,8667,0,8667
True,19156,558,19714
All,27823,558,28381



MISSING VALUES BY PLAYED STATUS


,total_players,missing_batting_order,missing_captain,missing_wicketkeeper
is_played,,,,
False,8667,8667,8667,8667
True,19714,11,0,0


In [6]:
match_ids = set(matches["match_id"])
ball_ids = set(ball_by_ball["match_id"])

print("Matches in match table:", len(match_ids))
print("Matches in ball data:", len(ball_ids))

print("Matches without ball data:", len(match_ids - ball_ids))
print("Ball data without match records:", len(ball_ids - match_ids))

print("\nMissing match IDs:")

display(
    matches[
        matches["match_id"].isin(match_ids - ball_ids)
    ][["match_id", "comp_id", "match_date", "result"]]
)

Matches in match table: 883
Matches in ball data: 867
Matches without ball data: 16
Ball data without match records: 0

Missing match IDs:


,match_id,comp_id,match_date,result
396,31281,2193,2024-05-13,Match Abandoned
399,31284,2193,2024-05-16,Match Abandoned
403,31288,2193,2024-05-19,Match Abandoned
425,32610,2354,2024-05-22,Match Abandoned
427,32612,2354,2024-05-28,Match Abandoned
447,32622,2398,2024-05-20,Match Abandoned
453,38797,2442,2024-12-14,Match Abandoned
481,36564,2548,2024-09-15,Match Abandoned
498,36210,2628,2025-09-14,Match Abandoned
593,40761,2725,2025-05-17,Match Abandoned


In [7]:
print("INNINGS DISTRIBUTION")

display(
    ball_by_ball.groupby("innings").agg(
        total_deliveries=("match_id", "size"),
        unique_matches=("match_id", "nunique"),
        unique_batters=("striker_id", "nunique"),
        unique_bowlers=("bowler_id", "nunique"),
        total_runs=("runs", "sum"),
        total_wickets=("is_wicket", "sum")
    )
)

INNINGS DISTRIBUTION


,total_deliveries,unique_matches,unique_batters,unique_bowlers,total_runs,total_wickets
innings,,,,,,
1,105727,867,778,564,143053,5630
2,95234,845,799,575,126453,5238
3,25,6,15,6,14,12
4,13,6,8,6,30,1


In [8]:
delivery_key = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num"
]

duplicate_deliveries = ball_by_ball[
    ball_by_ball.duplicated(
        subset=delivery_key,
        keep=False
    )
].sort_values(delivery_key)

print("Duplicate delivery records:",
      len(duplicate_deliveries))

display(duplicate_deliveries.head(30))

Duplicate delivery records: 0


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,...,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time


In [9]:
revised_cols = [
    "match_id",
    "match_date",
    "comp_id",
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target",
    "result"
]

revised = matches[
    matches["first_inn_revised_overs"].notna()
    | matches["second_inn_revised_overs"].notna()
    | matches["revised_target"].notna()
].copy()

print("Revised matches:", len(revised))

print("\nFirst innings revised values:")
display(revised["first_inn_revised_overs"].value_counts().sort_index())

print("\nSecond innings revised values:")
display(revised["second_inn_revised_overs"].value_counts().sort_index())

display(revised[revised_cols].head(40))

Revised matches: 34

First innings revised values:


first_inn_revised_overs
30.0     2
42.0     1
48.0     1
54.0     4
60.0     1
66.0     3
72.0     1
78.0     1
84.0     3
90.0     2
102.0    2
108.0    2
114.0    2
Name: count, dtype: int64


Second innings revised values:


second_inn_revised_overs
30.0     2
42.0     2
48.0     3
54.0     2
60.0     2
66.0     1
72.0     1
78.0     1
84.0     1
90.0     5
102.0    2
114.0    3
Name: count, dtype: int64

,match_id,match_date,comp_id,first_inn_revised_overs,second_inn_revised_overs,revised_target,result
8,13612,2022-02-11,1345,NaN,114.0,143.0,Australia Won by 20 runs (D/L method)
94,15410,2022-06-19,1439,114.0,NaN,NaN,Match Abandoned
95,15546,2022-07-02,1452,84.0,NaN,NaN,No Result
143,16753,2022-08-15,1543,66.0,NaN,NaN,Afghanistan Won by 27 Runs
144,16754,2022-08-17,1543,NaN,42.0,56.0,Ireland Won by 7 Wickets(D/L Method)
152,17395,2022-09-23,1584,48.0,48.0,91.0,India Won by 6 Wickets
177,17765,2022-10-14,1611,72.0,72.0,130.0,No Result
262,21586,2023-05-29,1715,NaN,90.0,171.0,Chennai Super Kings Won by 5 wickets (DLS met...
263,19959,2023-03-25,1718,66.0,NaN,NaN,West Indies won by 3 wickets
272,21070,2023-03-27,1769,NaN,48.0,104.0,Bangladesh Won by 22 Runs (DLS method)


In [10]:
other_extras = ball_by_ball[
    ball_by_ball["extras_type"]
    .astype("string")
    .str.lower()
    .eq("other")
    .fillna(False)
]

print("Total other extras:", len(other_extras))

display(other_extras[[
    "match_id", "innings", "overid",
    "striker_name", "bowler_name",
    "runs", "extras_type",
    "extras1_runs", "extras_2_runs",
    "is_wicket", "dismissal_type"
]].head(30))

Total other extras: 264


,match_id,innings,overid,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_wicket,dismissal_type
532,13513,1,10,Rovman Powell,Tymal Mills,0,other,1,0,False,NaN
1079,13515,1,16,Rovman Powell,Saqib Mahmood,0,other,1,0,False,NaN
1226,13515,2,19,Sam Billings,Sheldon Cottrell,2,other,1,0,False,NaN
2588,13614,1,18,Dasun Shanaka,Mitchell Starc,0,other,1,4,False,NaN
2609,13614,2,2,Ashton Agar,Nuwan Thushara,0,other,1,0,False,NaN
3309,14094,1,19,Mahendra Singh Dhoni,Shivam Mavi,6,other,1,0,False,NaN
4413,14098,2,20,Bhuvneshwar Kumar,Nathan Coulter Nile,0,other,1,0,False,NaN
5012,14101,1,17,Kagiso Rabada,Tim Southee,0,other,1,0,False,NaN
5365,14102,2,19,Kieron Pollard,Prasidh Krishna,0,other,1,0,True,RunOut
6575,14107,2,16,Patrick Cummins,Daniel Sams,2,other,1,0,False,NaN


In [11]:
oversteps = ball_by_ball[
    ball_by_ball["extras_type"] == "Overstep"
]

print("Total oversteps:", len(oversteps))

display(
    oversteps.groupby(
        ["runs", "extras1_runs", "extras_2_runs"],
        dropna=False
    ).size().reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(
    oversteps[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].head(20)
)

Total oversteps: 576


,runs,extras1_runs,extras_2_runs,count
0,0,1,0,218
3,1,1,0,174
6,4,1,0,81
7,6,1,0,47
4,2,1,0,33
1,0,1,1,12
2,0,1,4,9
5,3,1,0,2


,match_id,innings,overid,over_ball_num,innings_ball_num,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
495,13513,1,4,2,20,0,1,0,False,NaN
1229,13515,2,20,1,122,1,1,0,False,NaN
1485,13523,1,1,4,4,0,1,4,False,NaN
1486,13523,1,1,5,5,1,1,0,False,NaN
3191,14094,1,1,1,1,0,1,0,False,NaN
4169,14098,1,1,5,5,0,1,0,False,NaN
4181,14098,1,3,4,17,0,1,0,False,NaN
4187,14098,1,4,3,23,4,1,0,False,NaN
4192,14098,1,5,1,28,1,1,0,False,NaN
4452,14099,1,6,3,33,0,1,0,False,NaN


In [12]:
sequence_check = (
    ball_by_ball.groupby(["match_id", "innings"])
    .agg(
        total_deliveries=("innings_ball_num", "size"),
        unique_delivery_numbers=("innings_ball_num", "nunique"),
        minimum_delivery_number=("innings_ball_num", "min"),
        maximum_delivery_number=("innings_ball_num", "max")
    )
    .reset_index()
)

sequence_check["duplicate_numbers"] = (
    sequence_check["total_deliveries"]
    - sequence_check["unique_delivery_numbers"]
)

sequence_check["possible_gap"] = (
    sequence_check["maximum_delivery_number"]
    - sequence_check["minimum_delivery_number"] + 1
    > sequence_check["unique_delivery_numbers"]
)

print("Total innings checked:", len(sequence_check))

print("\nInnings with duplicate numbers:")
display(sequence_check[
    sequence_check["duplicate_numbers"] > 0
])

print("\nInnings with possible sequence gaps:")
display(sequence_check[
    sequence_check["possible_gap"]
])

print("\nMaximum innings ball number:",
      ball_by_ball["innings_ball_num"].max())

print("Maximum over ball number:",
      ball_by_ball["over_ball_num"].max())

Total innings checked: 1724

Innings with duplicate numbers:


,match_id,innings,total_deliveries,unique_delivery_numbers,minimum_delivery_number,maximum_delivery_number,duplicate_numbers,possible_gap



Innings with possible sequence gaps:


,match_id,innings,total_deliveries,unique_delivery_numbers,minimum_delivery_number,maximum_delivery_number,duplicate_numbers,possible_gap



Maximum innings ball number: 136
Maximum over ball number: 13


In [13]:
print("TEAM RUNS INVESTIGATION")

runs_check = ball_by_ball.copy()

runs_check["calculated_team_runs"] = (
    runs_check["runs"].fillna(0)
    + runs_check["extras1_runs"].fillna(0)
    + runs_check["extras_2_runs"].fillna(0)
)

print("Original columns:", ball_by_ball.shape[1])
print("Analysis columns:", runs_check.shape[1])

print("Batter runs:", runs_check["runs"].sum())
print("Extras 1:", runs_check["extras1_runs"].sum())
print("Extras 2:", runs_check["extras_2_runs"].sum())

print("Candidate total:",
      runs_check["calculated_team_runs"].sum())

display(
    runs_check.groupby("extras_type", dropna=False).agg(
        deliveries=("match_id", "size"),
        batter_runs=("runs", "sum"),
        candidate_team_runs=("calculated_team_runs", "sum")
    )
)

TEAM RUNS INVESTIGATION
Original columns: 33
Analysis columns: 34
Batter runs: 269550
Extras 1: 14623
Extras 2: 84
Candidate total: 284257


,deliveries,batter_runs,candidate_team_runs
extras_type,,,
Overstep,576,852,1476
Penalty,1,0,5
byes,529,0,1000
legbyes,2748,1,3604
other,264,412,711
wide,7714,0,9174
NaN,189167,268285,268287


In [14]:
import os

OUTPUT_PATH = "../outputs/data_audit/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 1. Missing values report
missing_rows = []

for name, df in datasets.items():
    for col in df.columns:
        missing_rows.append({
            "dataset": name,
            "column": col,
            "missing_count": df[col].isna().sum(),
            "missing_percentage": round(
                df[col].isna().mean() * 100, 2
            )
        })

pd.DataFrame(missing_rows).to_csv(
    OUTPUT_PATH + "missing_values_report.csv",
    index=False
)

# 2. Duplicate report
duplicate_rows = []

for name, df in datasets.items():
    duplicate_rows.append({
        "dataset": name,
        "total_rows": len(df),
        "duplicate_rows": df.duplicated().sum()
    })

pd.DataFrame(duplicate_rows).to_csv(
    OUTPUT_PATH + "duplicate_summary.csv",
    index=False
)

# 3. Identifier report
key_columns = {
    "competitions": ["comp_id"],
    "teams": ["team_id"],
    "matches": ["match_id"],
    "ball_by_ball": [
        "match_id", "innings", "overid", "over_ball_num"
    ],
    "player_mapping": ["comp_id", "team_id", "player_id"],
    "playing_eleven": ["match_id", "team_id", "player_id"]
}

key_rows = []

for name, keys in key_columns.items():
    df = datasets[name]

    key_rows.append({
        "dataset": name,
        "key_columns": ", ".join(keys),
        "missing_key_rows": df[keys].isna().any(axis=1).sum(),
        "duplicate_key_rows": df.duplicated(subset=keys).sum()
    })

pd.DataFrame(key_rows).to_csv(
    OUTPUT_PATH + "identifier_quality_report.csv",
    index=False
)

# 4. Matches without deliveries
delivery_ids = set(ball_by_ball["match_id"].dropna())

matches[
    ~matches["match_id"].isin(delivery_ids)
].to_csv(
    OUTPUT_PATH + "matches_without_deliveries.csv",
    index=False
)

# 5. Competition inventory
competitions.to_csv(
    OUTPUT_PATH + "competition_inventory.csv",
    index=False
)

# 6. Abandoned matches
abandoned = matches[
    matches["result"].astype("string")
    .str.contains("abandon", case=False, na=False)
]

abandoned.to_csv(
    OUTPUT_PATH + "abandoned_matches.csv",
    index=False
)

# 7. Revised matches
revised_matches = matches[
    matches["first_inn_revised_overs"].notna()
    | matches["second_inn_revised_overs"].notna()
    | matches["revised_target"].notna()
]

revised_matches.to_csv(
    OUTPUT_PATH + "revised_matches.csv",
    index=False
)

# 8. Innings distribution
innings_report = (
    ball_by_ball.groupby("innings")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        batters=("striker_id", "nunique"),
        bowlers=("bowler_id", "nunique"),
        recorded_runs=("runs", "sum"),
        wickets=("is_wicket", "sum")
    )
    .reset_index()
)

innings_report.to_csv(
    OUTPUT_PATH + "innings_distribution.csv",
    index=False
)

# 9. Dismissal report
dismissal_report = (
    ball_by_ball.groupby("dismissal_type", dropna=False)
    .size()
    .reset_index(name="count")
)

dismissal_report.to_csv(
    OUTPUT_PATH + "dismissal_report.csv",
    index=False
)

# 10. Extras report
extras_report = (
    ball_by_ball.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        runs=("runs", "sum"),
        extras1=("extras1_runs", "sum"),
        extras2=("extras_2_runs", "sum")
    )
    .reset_index()
)

extras_report.to_csv(
    OUTPUT_PATH + "extras_report.csv",
    index=False
)

# 11. Unknown extras
unknown_extras = ball_by_ball[
    ball_by_ball["extras_type"].isna()
    & (
        ball_by_ball["extras1_runs"].fillna(0).ne(0)
        | ball_by_ball["extras_2_runs"].fillna(0).ne(0)
    )
]

unknown_extras.to_csv(
    OUTPUT_PATH + "unknown_extras_records.csv",
    index=False
)

# 12. Delivery sequence
sequence_check.to_csv(
    OUTPUT_PATH + "delivery_sequence_report.csv",
    index=False
)

print("ALL AUDIT REPORTS GENERATED")
print("=" * 60)

for filename in sorted(os.listdir(OUTPUT_PATH)):
    print("SAVED:", filename)# Save duplicate delivery records separately

delivery_key = [
    "match_id",
    "innings",
    "overid",
    "over_ball_num"
]

duplicate_deliveries = ball_by_ball[
    ball_by_ball.duplicated(
        subset=delivery_key,
        keep=False
    )
].sort_values(delivery_key)

duplicate_deliveries.to_csv(
    OUTPUT_PATH + "duplicate_deliveries.csv",
    index=False
)

print(
    "Duplicate delivery records:",
    len(duplicate_deliveries)
)

ALL AUDIT REPORTS GENERATED
SAVED: abandoned_matches.csv
SAVED: competition_inventory.csv
SAVED: delivery_sequence_report.csv
SAVED: dismissal_report.csv
SAVED: duplicate_deliveries.csv
SAVED: duplicate_summary.csv
SAVED: extras_report.csv
SAVED: identifier_quality_report.csv
SAVED: innings_distribution.csv
SAVED: matches_without_deliveries.csv
SAVED: missing_values_detailed.csv
SAVED: missing_values_report.csv
SAVED: revised_matches.csv
SAVED: unknown_extras_records.csv
Duplicate delivery records: 0


In [15]:
print("=" * 65)
print("FINAL DATA QUALITY AUDIT SUMMARY")
print("=" * 65)

for name, df in datasets.items():

    print("\nDATASET:", name.upper())
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

print("\nAudit reports saved at:", OUTPUT_PATH)
print("Audit investigation completed.")

FINAL DATA QUALITY AUDIT SUMMARY

DATASET: COMPETITIONS
Rows: 141
Columns: 4
Missing values: 0
Duplicate rows: 0

DATASET: TEAMS
Rows: 353
Columns: 3
Missing values: 0
Duplicate rows: 0

DATASET: MATCHES
Rows: 883
Columns: 22
Missing values: 2736
Duplicate rows: 0

DATASET: BALL_BY_BALL
Rows: 200999
Columns: 33
Missing values: 766466
Duplicate rows: 0

DATASET: PLAYER_MAPPING
Rows: 6413
Columns: 6
Missing values: 0
Duplicate rows: 0

DATASET: PLAYING_ELEVEN
Rows: 28381
Columns: 9
Missing values: 26012
Duplicate rows: 0

Audit reports saved at: ../outputs/data_audit/
Audit investigation completed.


In [16]:
print("=" * 70)
print("MISSING VALUES BY DATASET")
print("=" * 70)

for name, df in datasets.items():

    print("\nDATASET:", name.upper())

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values")
    else:
        report = pd.DataFrame({
            "Missing Count": missing,
            "Percentage": (
                missing / len(df) * 100
            ).round(2)
        })

        display(report.sort_values(
            "Missing Count",
            ascending=False
        ))

MISSING VALUES BY DATASET

DATASET: COMPETITIONS
No missing values

DATASET: TEAMS
No missing values

DATASET: MATCHES


,Missing Count,Percentage
revised_target,860,97.40
first_inn_revised_overs,858,97.17
second_inn_revised_overs,858,97.17
away_team_captain_id,45,5.10
home_team_captain_id,44,4.98
win_team_id,41,4.64
toss_opted,15,1.70
toss_win_team_id,15,1.70



DATASET: BALL_BY_BALL


,Missing Count,Percentage
dismissal_type_id,190118,94.59
dismissal_type,190118,94.59
dismissed_player_id,190107,94.58
extras_type,189167,94.11
stroke,6158,3.06
over_or_around,407,0.20
delivery_style,391,0.19



DATASET: PLAYER_MAPPING
No missing values

DATASET: PLAYING_ELEVEN


,Missing Count,Percentage
batting_order,8678,30.58
is_captain,8667,30.54
is_wk,8667,30.54


In [17]:
missing_details = []

for name, df in datasets.items():

    for col in df.columns:

        count = df[col].isna().sum()

        if count > 0:

            missing_details.append({
                "dataset": name,
                "column": col,
                "missing_count": count,
                "missing_percentage": round(
                    count / len(df) * 100, 2
                )
            })

missing_details_df = pd.DataFrame(missing_details)

display(missing_details_df)

missing_details_df.to_csv(
    OUTPUT_PATH + "missing_values_detailed.csv",
    index=False
)

print("Detailed missing-value report saved.")

,dataset,column,missing_count,missing_percentage
0,matches,home_team_captain_id,44,4.98
1,matches,away_team_captain_id,45,5.10
2,matches,toss_win_team_id,15,1.70
3,matches,toss_opted,15,1.70
4,matches,first_inn_revised_overs,858,97.17
5,matches,second_inn_revised_overs,858,97.17
6,matches,revised_target,860,97.40
7,matches,win_team_id,41,4.64
8,ball_by_ball,dismissal_type_id,190118,94.59
9,ball_by_ball,dismissal_type,190118,94.59


Detailed missing-value report saved.


In [18]:
print("=" * 60)
print("FINAL ORIGINAL DATASET INTEGRITY CHECK")
print("=" * 60)

expected_shapes = {
    "competitions": (141, 4),
    "teams": (353, 3),
    "matches": (883, 22),
    "ball_by_ball": (200999, 33),
    "player_mapping": (6413, 6),
    "playing_eleven": (28381, 9)
}

for name, df in datasets.items():

    actual = df.shape
    expected = expected_shapes[name]

    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{name:20} | "
        f"Expected: {expected} | "
        f"Actual: {actual} | "
        f"{status}"
    )

FINAL ORIGINAL DATASET INTEGRITY CHECK
competitions         | Expected: (141, 4) | Actual: (141, 4) | PASS
teams                | Expected: (353, 3) | Actual: (353, 3) | PASS
matches              | Expected: (883, 22) | Actual: (883, 22) | PASS
ball_by_ball         | Expected: (200999, 33) | Actual: (200999, 33) | PASS
player_mapping       | Expected: (6413, 6) | Actual: (6413, 6) | PASS
playing_eleven       | Expected: (28381, 9) | Actual: (28381, 9) | PASS


In [20]:
# ==========================================
# CELL 19: CREATE WORKING COPIES
# ==========================================

import os
import pandas as pd
import numpy as np

# Create output directories
CLEANED_PATH = "../outputs/cleaned_data/"
REPORT_PATH = "../outputs/cleaning_reports/"

os.makedirs(CLEANED_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

# Create independent copies of original datasets
cleaned_datasets = {
    "competitions": competitions.copy(deep=True),
    "teams": teams.copy(deep=True),
    "matches": matches.copy(deep=True),
    "ball_by_ball": ball_by_ball.copy(deep=True),
    "player_mapping": player_mapping.copy(deep=True),
    "playing_eleven": playing_eleven.copy(deep=True)
}

# Display working copy information
print("=" * 70)
print("WORKING COPIES CREATED SUCCESSFULLY")
print("=" * 70)

for name, df in cleaned_datasets.items():

    print(f"\nDATASET: {name.upper()}")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

print("\nOriginal datasets remain separate and unchanged.")
print("Cleaning output directory:", CLEANED_PATH)
print("Cleaning reports directory:", REPORT_PATH)

WORKING COPIES CREATED SUCCESSFULLY

DATASET: COMPETITIONS
Rows: 141
Columns: 4
Missing values: 0
Duplicate rows: 0

DATASET: TEAMS
Rows: 353
Columns: 3
Missing values: 0
Duplicate rows: 0

DATASET: MATCHES
Rows: 883
Columns: 22
Missing values: 2736
Duplicate rows: 0

DATASET: BALL_BY_BALL
Rows: 200999
Columns: 33
Missing values: 766466
Duplicate rows: 0

DATASET: PLAYER_MAPPING
Rows: 6413
Columns: 6
Missing values: 0
Duplicate rows: 0

DATASET: PLAYING_ELEVEN
Rows: 28381
Columns: 9
Missing values: 26012
Duplicate rows: 0

Original datasets remain separate and unchanged.
Cleaning output directory: ../outputs/cleaned_data/
Cleaning reports directory: ../outputs/cleaning_reports/


In [21]:
# ==========================================
# CELL 20: DETAILED MISSING VALUES ANALYSIS
# ==========================================

missing_details = []

for name, df in cleaned_datasets.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        if missing_count > 0:

            missing_percentage = (
                missing_count / len(df)
            ) * 100

            missing_details.append({
                "dataset": name,
                "column": column,
                "total_rows": len(df),
                "missing_count": missing_count,
                "present_count": len(df) - missing_count,
                "missing_percentage": round(
                    missing_percentage, 2
                )
            })

missing_report = pd.DataFrame(missing_details)

print("=" * 80)
print("DETAILED MISSING VALUES REPORT")
print("=" * 80)

if not missing_report.empty:

    for name in cleaned_datasets.keys():

        print("\nDATASET:", name.upper())

        dataset_report = missing_report[
            missing_report["dataset"] == name
        ]

        if dataset_report.empty:
            print("No missing values.")
        else:
            display(
                dataset_report.sort_values(
                    "missing_count",
                    ascending=False
                )
            )

else:
    print("No missing values found.")

# Save report
missing_report.to_csv(
    REPORT_PATH + "missing_values_before_cleaning.csv",
    index=False
)

print("\nReport saved successfully.")

DETAILED MISSING VALUES REPORT

DATASET: COMPETITIONS
No missing values.

DATASET: TEAMS
No missing values.

DATASET: MATCHES


,dataset,column,total_rows,missing_count,present_count,missing_percentage
6,matches,revised_target,883,860,23,97.40
4,matches,first_inn_revised_overs,883,858,25,97.17
5,matches,second_inn_revised_overs,883,858,25,97.17
1,matches,away_team_captain_id,883,45,838,5.10
0,matches,home_team_captain_id,883,44,839,4.98
7,matches,win_team_id,883,41,842,4.64
3,matches,toss_opted,883,15,868,1.70
2,matches,toss_win_team_id,883,15,868,1.70



DATASET: BALL_BY_BALL


,dataset,column,total_rows,missing_count,present_count,missing_percentage
8,ball_by_ball,dismissal_type_id,200999,190118,10881,94.59
9,ball_by_ball,dismissal_type,200999,190118,10881,94.59
10,ball_by_ball,dismissed_player_id,200999,190107,10892,94.58
11,ball_by_ball,extras_type,200999,189167,11832,94.11
12,ball_by_ball,stroke,200999,6158,194841,3.06
14,ball_by_ball,over_or_around,200999,407,200592,0.20
13,ball_by_ball,delivery_style,200999,391,200608,0.19



DATASET: PLAYER_MAPPING
No missing values.

DATASET: PLAYING_ELEVEN


,dataset,column,total_rows,missing_count,present_count,missing_percentage
15,playing_eleven,batting_order,28381,8678,19703,30.58
16,playing_eleven,is_captain,28381,8667,19714,30.54
17,playing_eleven,is_wk,28381,8667,19714,30.54



Report saved successfully.


In [23]:
# ==========================================
# CELL 21: MISSING VALUE RELATIONSHIP CHECK
# ==========================================

print("=" * 75)
print("MISSING VALUE RELATIONSHIP INVESTIGATION")
print("=" * 75)


# 1. PLAYING ELEVEN INVESTIGATION

print("\n1. PLAYING ELEVEN")

print("\nPlaying status distribution:")
display(
    cleaned_datasets["playing_eleven"]["is_played"]
    .value_counts(dropna=False)
)

playing = cleaned_datasets["playing_eleven"]

print("\nMissing values grouped by playing status:")

display(
    playing.groupby("is_played", dropna=False)
    .agg(
        total_players=("player_id", "size"),
        missing_batting_order=("batting_order", lambda x: x.isna().sum()),
        missing_captain=("is_captain", lambda x: x.isna().sum()),
        missing_wicketkeeper=("is_wk", lambda x: x.isna().sum())
    )
)


# 2. DISMISSAL INVESTIGATION

print("\n2. DISMISSAL FIELDS")

deliveries = cleaned_datasets["ball_by_ball"]

display(
    deliveries.groupby("is_wicket", dropna=False)
    .agg(
        total_deliveries=("match_id", "size"),
        missing_dismissal_type=("dismissal_type", lambda x: x.isna().sum()),
        missing_dismissed_player=("dismissed_player_id", lambda x: x.isna().sum()),
        present_dismissed_player=("dismissed_player_id", lambda x: x.notna().sum())
    )
)

print("\nDismissed player present but is_wicket is not 1:")

display(
    deliveries[
        deliveries["dismissed_player_id"].notna()
        & (deliveries["is_wicket"] != 1)
    ][[
        "match_id", "innings", "striker_name",
        "dismissed_player_id", "dismissal_type", "is_wicket"
    ]].head(30)
)


# 3. EXTRAS INVESTIGATION

print("\n3. EXTRAS FIELDS")

extras = deliveries.copy()

extras["total_extras_recorded"] = (
    extras["extras1_runs"].fillna(0)
    + extras["extras_2_runs"].fillna(0)
)

print("\nMissing extras_type but extras runs present:")

display(
    extras[
        extras["extras_type"].isna()
        & (extras["total_extras_recorded"] > 0)
    ][[
        "match_id", "innings", "runs",
        "extras_type", "extras1_runs", "extras_2_runs"
    ]].head(30)
)

print("\nExtras type present but both extras values are zero:")

display(
    extras[
        extras["extras_type"].notna()
        & (extras["total_extras_recorded"] == 0)
    ][[
        "match_id", "innings", "runs",
        "extras_type", "extras1_runs", "extras_2_runs"
    ]].head(30)
)


# 4. MATCH RESULT INVESTIGATION

print("\n4. MATCH WINNER INFORMATION")

match_data = cleaned_datasets["matches"]

display(
    match_data[
        match_data["win_team_id"].isna()
    ][[
        "match_id", "match_date", "home_team",
        "away_team", "result", "win_team_id"
    ]].head(50)
)


# 5. MATCH TOSS INVESTIGATION

print("\n5. TOSS INFORMATION")

display(
    match_data[
        match_data["toss_win_team_id"].isna()
        | match_data["toss_opted"].isna()
    ][[
        "match_id", "match_date", "home_team",
        "away_team", "result",
        "toss_win_team_id", "toss_opted"
    ]]
)


print("\nMISSING VALUE RELATIONSHIP CHECK COMPLETED")

MISSING VALUE RELATIONSHIP INVESTIGATION

1. PLAYING ELEVEN

Playing status distribution:


is_played
True     19714
False     8667
Name: count, dtype: int64


Missing values grouped by playing status:


,total_players,missing_batting_order,missing_captain,missing_wicketkeeper
is_played,,,,
False,8667,8667,8667,8667
True,19714,11,0,0



2. DISMISSAL FIELDS


,total_deliveries,missing_dismissal_type,missing_dismissed_player,present_dismissed_player
is_wicket,,,,
False,190118,190118,190106,12
True,10881,0,1,10880


... (7 outputs left out)

,match_id,match_date,home_team,away_team,result,win_team_id
94,15410,2022-06-19,India,South Africa,Match Abandoned,NaN
95,15546,2022-07-02,West Indies,Bangladesh,No Result,NaN
177,17765,2022-10-14,Australia,England,No Result,NaN
179,18532,2022-11-22,New Zealand,India,Match Tied,NaN
234,20226,2023-05-03,Lucknow Super Giants,Chennai Super Kings,Match Abandoned,NaN
281,21406,2023-04-20,Pakistan,New Zealand,Match Abandoned,NaN
315,28333,2023-12-29,New Zealand,Bangladesh,No Result,NaN
396,31281,2024-05-13,Gujarat Titans,Kolkata Knight Riders,Match Abandoned,NaN
399,31284,2024-05-16,Sunrisers Hyderabad,Gujarat Titans,Match Abandoned,NaN
403,31288,2024-05-19,Rajasthan Royals,Kolkata Knight Riders,Match Abandoned,NaN



5. TOSS INFORMATION


,match_id,match_date,home_team,away_team,result,toss_win_team_id,toss_opted
396,31281,2024-05-13,Gujarat Titans,Kolkata Knight Riders,Match Abandoned,NaN,NaN
399,31284,2024-05-16,Sunrisers Hyderabad,Gujarat Titans,Match Abandoned,NaN,NaN
425,32610,2024-05-22,England,Pakistan,Match Abandoned,NaN,NaN
427,32612,2024-05-28,England,Pakistan,Match Abandoned,NaN,NaN
447,32622,2024-05-20,Ireland,Scotland,Match Abandoned,NaN,NaN
453,38797,2024-12-14,South Africa,Pakistan,Match Abandoned,NaN,NaN
481,36564,2024-09-15,England,Australia,Match Abandoned,NaN,NaN
498,36210,2025-09-14,England,South Africa,Match Abandoned,NaN,NaN
593,40761,2025-05-17,Royal Challengers Bengaluru,Kolkata Knight Riders,Match Abandoned,NaN,NaN
615,41502,2025-06-12,Ireland,West Indies,Match Abandoned,NaN,NaN



MISSING VALUE RELATIONSHIP CHECK COMPLETED


In [24]:
playing = cleaned_datasets["playing_eleven"]

# Show the 11 players who played but have no batting order
missing_batting_order = playing[
    (playing["is_played"] == True)
    & (playing["batting_order"].isna())
]

print("Number of affected records:", len(missing_batting_order))

display(
    missing_batting_order[
        [
            "match_id",
            "team_id",
            "player_id",
            "is_played",
            "batting_order",
            "is_captain",
            "is_wk",
            "is_impact"
        ]
    ]
)

Number of affected records: 11


,match_id,team_id,player_id,is_played,batting_order,is_captain,is_wk,is_impact
22883,48512,25,7690,True,NaN,False,False,0
22884,48512,25,3091,True,NaN,False,False,0
22885,48512,25,20327,True,NaN,False,True,0
22886,48512,25,7696,True,NaN,False,False,0
22887,48512,25,20704,True,NaN,False,False,0
22888,48512,25,2921,True,NaN,False,False,0
22889,48512,25,2490,True,NaN,False,False,0
22890,48512,25,5979,True,NaN,False,False,0
22891,48512,25,4926,True,NaN,False,False,0
22892,48512,25,5189,True,NaN,False,False,0


In [25]:
# Investigate match 48512

match_id = 48512

print("MATCH DETAILS")
display(
    cleaned_datasets["matches"][
        cleaned_datasets["matches"]["match_id"] == match_id
    ]
)

print("\nPLAYING XI RECORDS")
display(
    cleaned_datasets["playing_eleven"][
        cleaned_datasets["playing_eleven"]["match_id"] == match_id
    ]
)

print("\nBALL-BY-BALL RECORDS")
display(
    cleaned_datasets["ball_by_ball"][
        cleaned_datasets["ball_by_ball"]["match_id"] == match_id
    ].head(20)
)

MATCH DETAILS


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,...,away_team_captain_id,toss_win_team_id,toss_opted,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result
824,48512,3317,1,2026-01-07,Others,60,Sri Lanka,27,Sri Lanka,25,...,3489.0,25.0,Fielding,4926,NaN,NaN,NaN,0,25.0,Pakistan Won by 6 wickets



PLAYING XI RECORDS


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
22879,48512,25,37879,NaN,False,NaN,NaN,0,0
22880,48512,25,6922,NaN,False,NaN,NaN,0,0
22881,48512,25,52338,NaN,False,NaN,NaN,0,0
22882,48512,25,8049,NaN,False,NaN,NaN,0,0
22883,48512,25,7690,NaN,True,False,False,0,0
22884,48512,25,3091,NaN,True,False,False,0,0
22885,48512,25,20327,NaN,True,False,True,0,0
22886,48512,25,7696,NaN,True,False,False,0,0
22887,48512,25,20704,NaN,True,False,False,0,0
22888,48512,25,2921,NaN,True,False,False,0,0



BALL-BY-BALL RECORDS


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,...,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time
170667,48512,1,1,1,1,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,132.0,Push,Inswing,Over,Off-side,2026-01-07 13:32:55 UTC
170668,48512,1,1,2,2,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,136.0,Cover Drive,Inswing,Over,Off-side,2026-01-07 13:34:09 UTC
170669,48512,1,1,3,3,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,139.0,Backfoot defensive,Inswing,Over,Off-side,2026-01-07 13:34:46 UTC
170670,48512,1,1,4,4,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,127.0,Cover Drive,Off Cutter,Over,Off-side,2026-01-07 13:35:25 UTC
170671,48512,1,1,5,5,27,25,7049,Kamil Mishara,6931,...,legbyes,1,0,Pace,137.0,Flick,Inswing,Over,On-side,2026-01-07 13:36:11 UTC
170672,48512,1,1,6,6,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,125.0,Worked,Off Cutter,Over,On-side,2026-01-07 13:36:41 UTC
170673,48512,1,2,1,7,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,135.0,Backfoot defensive,Out Swing,Over,Off-side,2026-01-07 13:37:59 UTC
170674,48512,1,2,2,8,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,139.0,Frontfoot defensive,Out Swing,Over,Off-side,2026-01-07 13:38:28 UTC
170675,48512,1,2,3,9,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,137.0,Pull shot,Seamed In,Over,On-side,2026-01-07 13:39:11 UTC
170676,48512,1,2,4,10,27,25,6931,Pathum Nissanka,7049,...,NaN,0,0,Pace,138.0,Square Cut,Seamed Out,Over,Off-side,2026-01-07 13:39:44 UTC


In [26]:
# Check actual batting order from ball-by-ball data for match 48512

deliveries = cleaned_datasets["ball_by_ball"]

match_deliveries = deliveries[
    (deliveries["match_id"] == 48512)
    & (deliveries["innings"] == 1)
]

actual_batting_order = (
    match_deliveries
    .groupby(["striker_id", "striker_name"], as_index=False)
    .agg(
        first_ball_faced=("innings_ball_num", "min"),
        total_runs=("runs", "sum"),
        balls_recorded=("innings_ball_num", "count")
    )
    .sort_values("first_ball_faced")
)

display(actual_batting_order)

,striker_id,striker_name,first_ball_faced,total_runs,balls_recorded
6,6931,Pathum Nissanka,1,12,12
7,7049,Kamil Mishara,5,0,3
3,3672,Kusal Mendis,15,14,15
1,2407,Dhananjaya de Silva,21,10,10
4,4895,Charith Asalanka,40,18,15
10,26519,Janith Liyanage,42,40,31
5,4900,Wanindu Hasaranga,70,18,12
2,2436,Dasun Shanaka,93,12,9
0,2399,Dushmantha Chameera,107,0,3
9,19316,Maheesh Theekshana,110,1,3


In [27]:
# Create a separate report for the reconstructed batting order

reconstructed_order = actual_batting_order[
    ["striker_id", "striker_name", "first_ball_faced"]
].copy()

reconstructed_order["reconstructed_batting_order"] = range(
    1, len(reconstructed_order) + 1
)

reconstructed_order["match_id"] = 48512

reconstructed_order = reconstructed_order[
    [
        "match_id",
        "striker_id",
        "striker_name",
        "reconstructed_batting_order",
        "first_ball_faced"
    ]
]

display(reconstructed_order)

reconstructed_order.to_csv(
    REPORT_PATH + "match_48512_reconstructed_batting_order.csv",
    index=False
)

print("Reconstruction report saved successfully.")

,match_id,striker_id,striker_name,reconstructed_batting_order,first_ball_faced
6,48512,6931,Pathum Nissanka,1,1
7,48512,7049,Kamil Mishara,2,5
3,48512,3672,Kusal Mendis,3,15
1,48512,2407,Dhananjaya de Silva,4,21
4,48512,4895,Charith Asalanka,5,40
10,48512,26519,Janith Liyanage,6,42
5,48512,4900,Wanindu Hasaranga,7,70
2,48512,2436,Dasun Shanaka,8,93
0,48512,2399,Dushmantha Chameera,9,107
9,48512,19316,Maheesh Theekshana,10,110


Reconstruction report saved successfully.


In [28]:
# Investigate the wicket record with missing dismissed player ID

deliveries = cleaned_datasets["ball_by_ball"]

missing_dismissed_player = deliveries[
    (deliveries["is_wicket"] == True)
    & (deliveries["dismissed_player_id"].isna())
]

print("Records found:", len(missing_dismissed_player))

display(missing_dismissed_player.T)

Records found: 1


,127899
match_id,40709
innings,2
overid,19
over_ball_num,5
innings_ball_num,117
batting_team_id,2
bowling_team_id,3509
striker_id,3330
striker_name,Hardik Pandya
non_striker_id,7897


In [29]:
# Verify the Retire Out record against the match data

match_id = 40709

print("MATCH DETAILS")
display(
    cleaned_datasets["matches"][
        cleaned_datasets["matches"]["match_id"] == match_id
    ]
)

print("\nALL WICKET RECORDS IN THIS MATCH")
display(
    cleaned_datasets["ball_by_ball"][
        (cleaned_datasets["ball_by_ball"]["match_id"] == match_id)
        & (cleaned_datasets["ball_by_ball"]["is_wicket"] == True)
    ][
        [
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "dismissal_type",
            "dismissed_player_id"
        ]
    ]
)

MATCH DETAILS


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,...,away_team_captain_id,toss_win_team_id,toss_opted,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result
541,40709,2725,16,2025-04-04,League,3272,India,3509,Lucknow Super Giants,2,...,3330.0,2.0,Fielding,58134,NaN,NaN,NaN,0,3509.0,Lucknow Super Giants Won by 12 runs



ALL WICKET RECORDS IN THIS MATCH


,innings,overid,over_ball_num,striker_name,dismissal_type,dismissed_player_id
127698,1,7,6,Mitchell Marsh,Caught,818.0
127710,1,9,6,Nicholas Pooran,Caught,3125.0
127721,1,11,4,Rishabh Pant,Caught,4703.0
127754,1,16,6,Ayush Badoni,Caught,6460.0
127768,1,18,6,Aiden Markram,Caught,3605.0
127772,1,19,3,Abdul Samad,Caught,7446.0
127779,1,20,4,David Miller,Caught,1077.0
127780,1,20,5,Akash Deep,Caught,7473.0
127792,2,2,4,William Jacks,Caught,6272.0
127796,2,3,2,Ryan Rickelton,Caught,4007.0


In [30]:
# Investigate extras inconsistencies

deliveries = cleaned_datasets["ball_by_ball"]

print("CASE 1: Missing extras type but extras runs are present")

case_1 = deliveries[
    deliveries["extras_type"].isna()
    & (
        deliveries["extras1_runs"].fillna(0)
        + deliveries["extras_2_runs"].fillna(0)
        > 0
    )
]

display(case_1[
    [
        "match_id", "innings", "overid",
        "over_ball_num", "striker_name",
        "runs", "extras_type",
        "extras1_runs", "extras_2_runs"
    ]
])

print("\nCASE 2: Extras type present but extras runs are zero")

case_2 = deliveries[
    deliveries["extras_type"].notna()
    & (
        deliveries["extras1_runs"].fillna(0)
        + deliveries["extras_2_runs"].fillna(0)
        == 0
    )
]

display(case_2[
    [
        "match_id", "innings", "overid",
        "over_ball_num", "striker_name",
        "runs", "extras_type",
        "extras1_runs", "extras_2_runs"
    ]
])

print("\nCASE 3: Extras type distribution")

display(
    deliveries["extras_type"].value_counts(dropna=False)
)

print("\nEXTRAS INVESTIGATION COMPLETED")

CASE 1: Missing extras type but extras runs are present


,match_id,innings,overid,over_ball_num,striker_name,runs,extras_type,extras1_runs,extras_2_runs
36854,17555,1,5,4,Dawid Malan,0,NaN,0,1
107489,36097,2,8,6,Rassie van der Dussen,0,NaN,1,0



CASE 2: Extras type present but extras runs are zero


,match_id,innings,overid,over_ball_num,striker_name,runs,extras_type,extras1_runs,extras_2_runs
113765,37099,1,14,1,Nimesh Vimukthi,1,legbyes,0,0



CASE 3: Extras type distribution


extras_type
NaN         189167
wide          7714
legbyes       2748
Overstep       576
byes           529
other          264
Penalty          1
Name: count, dtype: int64


EXTRAS INVESTIGATION COMPLETED


In [31]:
# Inspect complete records for the three extras inconsistencies

deliveries = cleaned_datasets["ball_by_ball"]

suspect_indices = [36854, 107489, 113765]

for idx in suspect_indices:
    print(f"\n{'=' * 60}")
    print(f"RECORD INDEX: {idx}")
    display(deliveries.loc[[idx]].T)


RECORD INDEX: 36854


,36854
match_id,17555
innings,1
overid,5
over_ball_num,4
innings_ball_num,29
batting_team_id,22
bowling_team_id,25
striker_id,728
striker_name,Dawid Malan
non_striker_id,6272



RECORD INDEX: 107489


,107489
match_id,36097
innings,2
overid,8
over_ball_num,6
innings_ball_num,49
batting_team_id,19
bowling_team_id,28
striker_id,2930
striker_name,Rassie van der Dussen
non_striker_id,20702



RECORD INDEX: 113765


,113765
match_id,37099
innings,1
overid,14
over_ball_num,1
innings_ball_num,82
batting_team_id,3674
bowling_team_id,4444
striker_id,26514
striker_name,Nimesh Vimukthi
non_striker_id,7155


In [32]:
# Document decisions for investigated missing values

cleaning_decisions = pd.DataFrame([
    {
        "dataset": "playing_eleven",
        "issue": "11 missing batting orders in match 48512",
        "decision": "Preserve original values; save reconstructed order separately",
        "reason": "Order inferred from first appearance in ball-by-ball data"
    },
    {
        "dataset": "ball_by_ball",
        "issue": "1 missing dismissed_player_id for Retire Out",
        "decision": "Preserve missing value",
        "reason": "Dismissal is recorded as Retire Out"
    },
    {
        "dataset": "ball_by_ball",
        "issue": "2 missing extras_type with extras runs present",
        "decision": "Preserve missing values",
        "reason": "Extra classification cannot be reliably inferred"
    },
    {
        "dataset": "ball_by_ball",
        "issue": "1 legbyes record with zero extras values",
        "decision": "Preserve original values",
        "reason": "Source inconsistency requires further verification"
    },
    {
        "dataset": "matches",
        "issue": "Missing winner IDs for abandoned, no-result or tied matches",
        "decision": "Preserve missing values",
        "reason": "No winner should be assigned"
    },
    {
        "dataset": "playing_eleven",
        "issue": "Missing playing details for non-playing players",
        "decision": "Preserve missing values",
        "reason": "Missing values are associated with is_played=False"
    }
])

display(cleaning_decisions)

cleaning_decisions.to_csv(
    REPORT_PATH + "cleaning_decisions.csv",
    index=False
)

print("Cleaning decision report saved successfully.")

,dataset,issue,decision,reason
0,playing_eleven,11 missing batting orders in match 48512,Preserve original values; save reconstructed o...,Order inferred from first appearance in ball-b...
1,ball_by_ball,1 missing dismissed_player_id for Retire Out,Preserve missing value,Dismissal is recorded as Retire Out
2,ball_by_ball,2 missing extras_type with extras runs present,Preserve missing values,Extra classification cannot be reliably inferred
3,ball_by_ball,1 legbyes record with zero extras values,Preserve original values,Source inconsistency requires further verifica...
4,matches,"Missing winner IDs for abandoned, no-result or...",Preserve missing values,No winner should be assigned
5,playing_eleven,Missing playing details for non-playing players,Preserve missing values,Missing values are associated with is_played=F...


Cleaning decision report saved successfully.


In [33]:
# Check data types in all six working datasets

for name, df in cleaned_datasets.items():
    print("\n" + "=" * 60)
    print(f"DATASET: {name}")
    print("=" * 60)
    display(
        pd.DataFrame({
            "column": df.columns,
            "data_type": df.dtypes.astype(str).values,
            "missing_values": df.isna().sum().values
        })
    )


DATASET: competitions


,column,data_type,missing_values
0,comp_id,int64,0
1,comp_name,str,0
2,match_format,str,0
3,season,str,0



DATASET: teams


,column,data_type,missing_values
0,comp_id,int64,0
1,team_id,int64,0
2,team_name,str,0



DATASET: matches


,column,data_type,missing_values
0,match_id,int64,0
1,comp_id,int64,0
2,match_number,int64,0
3,match_date,str,0
4,stage,str,0
5,venue_id,int64,0
6,venue_country,str,0
7,home_team_id,int64,0
8,home_team,str,0
9,away_team_id,int64,0


... (2 outputs left out)


DATASET: player_mapping


,column,data_type,missing_values
0,comp_id,int64,0
1,comp_name,str,0
2,team_id,int64,0
3,team_name,str,0
4,player_id,int64,0
5,player_name,str,0



DATASET: playing_eleven


,column,data_type,missing_values
0,match_id,int64,0
1,team_id,int64,0
2,player_id,int64,0
3,batting_order,float64,8678
4,is_played,bool,0
5,is_captain,object,8667
6,is_wk,object,8667
7,is_impact,int64,0
8,is_impact_off,int64,0


In [34]:
# DATE FORMAT INVESTIGATION

print("MATCH DATE EXAMPLES")
display(cleaned_datasets["matches"]["match_date"].head(10))

print("\nMATCH DATE FORMAT")
print(cleaned_datasets["matches"]["match_date"].dtype)

print("\nDELIVERY TIMESTAMP EXAMPLES")
display(cleaned_datasets["ball_by_ball"]["delivery_ist_time"].head(10))

print("\nDELIVERY TIMESTAMP FORMAT")
print(cleaned_datasets["ball_by_ball"]["delivery_ist_time"].dtype)

print("\nINVALID MATCH DATES")
match_dates = pd.to_datetime(
    cleaned_datasets["matches"]["match_date"],
    errors="coerce"
)
print("Invalid dates:", match_dates.isna().sum())

print("\nINVALID DELIVERY TIMESTAMPS")
delivery_dates = pd.to_datetime(
    cleaned_datasets["ball_by_ball"]["delivery_ist_time"],
    errors="coerce",
    utc=True
)
print("Invalid timestamps:", delivery_dates.isna().sum())

MATCH DATE EXAMPLES


0    2022-01-22
1    2022-01-23
2    2022-01-26
3    2022-01-29
4    2022-01-30
5    2022-02-16
6    2022-02-18
7    2022-02-20
8    2022-02-11
9    2022-02-13
Name: match_date, dtype: str


MATCH DATE FORMAT
str

DELIVERY TIMESTAMP EXAMPLES


0    2022-01-23 00:00:00 UTC
1    2022-01-23 00:00:00 UTC
2    2022-01-23 00:00:00 UTC
3    2022-01-23 00:00:00 UTC
4    2022-01-23 00:00:00 UTC
5    2022-01-23 00:00:00 UTC
6    2022-01-23 00:00:00 UTC
7    2022-01-23 00:00:00 UTC
8    2022-01-23 00:00:00 UTC
9    2022-01-23 00:00:00 UTC
Name: delivery_ist_time, dtype: str


DELIVERY TIMESTAMP FORMAT
str

INVALID MATCH DATES
Invalid dates: 0

INVALID DELIVERY TIMESTAMPS
Invalid timestamps: 0


In [35]:
# Convert date columns in working copies only

cleaned_datasets["matches"]["match_date"] = pd.to_datetime(
    cleaned_datasets["matches"]["match_date"],
    errors="raise"
)

cleaned_datasets["ball_by_ball"]["delivery_ist_time"] = pd.to_datetime(
    cleaned_datasets["ball_by_ball"]["delivery_ist_time"],
    errors="raise",
    utc=True
)

print("DATE CONVERSION COMPLETED")

print(
    "Match date type:",
    cleaned_datasets["matches"]["match_date"].dtype
)

print(
    "Delivery timestamp type:",
    cleaned_datasets["ball_by_ball"]["delivery_ist_time"].dtype
)

print("Original source datasets remain unchanged.")

DATE CONVERSION COMPLETED
Match date type: datetime64[us]
Delivery timestamp type: datetime64[us, UTC]
Original source datasets remain unchanged.


In [36]:
# Investigate playing status and boolean columns

playing = cleaned_datasets["playing_eleven"]

print("IS_CAPTAIN VALUES")
display(playing["is_captain"].value_counts(dropna=False))

print("\nIS_WK VALUES")
display(playing["is_wk"].value_counts(dropna=False))

print("\nIS_PLAYED VALUES")
display(playing["is_played"].value_counts(dropna=False))

print("\nBOOLEAN COLUMN DATA TYPES")
print(playing[["is_played", "is_captain", "is_wk"]].dtypes)

print("\nCHECK MISSING VALUES FOR PLAYED PLAYERS")
display(
    playing[playing["is_played"] == True][
        ["is_captain", "is_wk"]
    ].isna().sum()
)

IS_CAPTAIN VALUES


is_captain
False    18031
NaN       8667
True      1683
Name: count, dtype: int64


IS_WK VALUES


is_wk
False    18039
NaN       8667
True      1675
Name: count, dtype: int64


IS_PLAYED VALUES


is_played
True     19714
False     8667
Name: count, dtype: int64


BOOLEAN COLUMN DATA TYPES
is_played       bool
is_captain    object
is_wk         object
dtype: object

CHECK MISSING VALUES FOR PLAYED PLAYERS


is_captain    0
is_wk         0
dtype: int64

In [37]:
# Convert boolean columns while preserving missing values

playing = cleaned_datasets["playing_eleven"]

playing["is_captain"] = playing["is_captain"].astype("boolean")
playing["is_wk"] = playing["is_wk"].astype("boolean")

print("BOOLEAN CONVERSION COMPLETED")

print("\nUpdated data types:")
print(playing[["is_played", "is_captain", "is_wk"]].dtypes)

print("\nMissing values after conversion:")
print(playing[["is_captain", "is_wk"]].isna().sum())

BOOLEAN CONVERSION COMPLETED

Updated data types:
is_played        bool
is_captain    boolean
is_wk         boolean
dtype: object

Missing values after conversion:
is_captain    8667
is_wk         8667
dtype: int64


In [38]:
# Validate working datasets after type conversions

print("FINAL WORKING DATASET VALIDATION")
print("=" * 60)

for name, df in cleaned_datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Total missing values:", df.isna().sum().sum())

print("\nDATE TYPES")
print(cleaned_datasets["matches"]["match_date"].dtype)
print(cleaned_datasets["ball_by_ball"]["delivery_ist_time"].dtype)

print("\nBOOLEAN TYPES")
print(
    cleaned_datasets["playing_eleven"][
        ["is_played", "is_captain", "is_wk"]
    ].dtypes
)

FINAL WORKING DATASET VALIDATION

competitions
Shape: (141, 4)
Duplicate rows: 0
Total missing values: 0

teams
Shape: (353, 3)
Duplicate rows: 0
Total missing values: 0

matches
Shape: (883, 22)
Duplicate rows: 0
Total missing values: 2736

ball_by_ball
Shape: (200999, 33)
Duplicate rows: 0
Total missing values: 766466

player_mapping
Shape: (6413, 6)
Duplicate rows: 0
Total missing values: 0

playing_eleven
Shape: (28381, 9)
Duplicate rows: 0
Total missing values: 26012

DATE TYPES
datetime64[us]
datetime64[us, UTC]

BOOLEAN TYPES
is_played        bool
is_captain    boolean
is_wk         boolean
dtype: object


In [39]:
# Save cleaned working datasets

for name, df in cleaned_datasets.items():
    output_file = CLEANED_PATH + name + "_cleaned.csv"
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

print("\nALL SIX CLEANED DATASETS SAVED SUCCESSFULLY")

Saved: ../outputs/cleaned_data/competitions_cleaned.csv
Saved: ../outputs/cleaned_data/teams_cleaned.csv
Saved: ../outputs/cleaned_data/matches_cleaned.csv
Saved: ../outputs/cleaned_data/ball_by_ball_cleaned.csv
Saved: ../outputs/cleaned_data/player_mapping_cleaned.csv
Saved: ../outputs/cleaned_data/playing_eleven_cleaned.csv

ALL SIX CLEANED DATASETS SAVED SUCCESSFULLY
